In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from itertools import product
import math
import scienceplots

plt.style.use(['science','nature'])
plt.rcParams.update({'figure.dpi': '300'})

from scipy import signal
from scipy import stats
from scipy.interpolate import make_smoothing_spline

# Be careful of 2pi errors.

def rpm(rad_per_second):
    return (rad_per_second / (2 * math.pi)) * 60

def rad_per_sec(rpm_input):
    return (rpm_input * (2 * math.pi)) / 60

e0 = 8.854 * 10 **-12

df_27_kV = pd.read_csv('20260817_processed_video_csv/27.0_kV.csv', delimiter=",", names=["time(s)", "frame", "x", "y", "r", "theta", "omega(deg/sec)", "alpha"])
df_27_kV = df_27_kV.iloc[2:]
df_27_kV = df_27_kV.reset_index(drop=True)
df_27_kV = df_27_kV.apply(pd.to_numeric, errors='coerce')


df_28_kV = pd.read_csv('20260817_processed_video_csv/28.0_kV.csv', delimiter=",", names=["time(s)", "frame", "x", "y", "r", "theta", "omega(deg/sec)", "alpha"])
df_28_kV = df_28_kV.iloc[2:]
df_28_kV = df_28_kV.reset_index(drop=True)
df_28_kV = df_28_kV.apply(pd.to_numeric, errors='coerce')


df_29_kV = pd.read_csv('20260817_processed_video_csv/29.0_kV_wrong_direction.csv', delimiter=",", names=["time(s)", "frame", "x", "y", "r", "theta", "omega(deg/sec)", "alpha"])
df_29_kV = df_29_kV.iloc[2:]
df_29_kV = df_29_kV.reset_index(drop=True)
df_29_kV = df_29_kV.apply(pd.to_numeric, errors='coerce')

df_30_kV = pd.read_csv('20260817_processed_video_csv/30kV_spindown.csv', delimiter=",", names=["time(s)", "frame", "x", "y", "r", "theta", "omega(deg/sec)", "alpha"])
df_30_kV = df_30_kV.iloc[2:]
df_30_kV = df_30_kV.reset_index(drop=True)
df_30_kV = df_30_kV.apply(pd.to_numeric, errors='coerce')



print(df_29_kV)




cutoff_freq = 99
sampling_rate = 200
order = 3




m_rotor = 0.00140 + 0.0002 # kg 11.5 grams plust he tape.
r_rotor = 0.029 # m

m_rotor_supports = 0.00685
r_rotor_supports = 0.029

m_shaft = 0.002
r_shaft = 0.003
I_rotor =  m_rotor * r_rotor**2 + 0.5 * m_rotor_supports * r_rotor_supports**2 + 0.5 *2 * m_shaft *r_shaft**2
print(I_rotor)

dataframes = [df_27_kV, df_28_kV, df_29_kV, df_30_kV]





def coordinate_transform_and_filter(df, window=200):
    df["x_c"] = np.NAN
    df["y_c"] = np.NAN
    for i in range(len(df) - window):
        df["x_c"][i] = (max(df["x"][i:i+window]) + min(df["x"][i:i+window])) / 2
        df['y_c'][i] = (max(df["y"][i:i+window]) + min(df["y"][i:i+window])) / 2
    df['x_c'].ffill(inplace=True)
    df['y_c'].ffill(inplace=True)
    df["theta"] = np.arctan2( (df["y"] - df['y_c']), (df["x"] - df['x_c']))
    df["omega(rad/s)"] =  -df["theta"].diff(periods=-1) / df["dt(s)"]
    return df


t = df_30_kV["time(s)"]
print("NaNs in time:", t.isna().sum())
print("Non-positive diffs:", (t.diff().dropna() <= 0).sum())
print("Duplicated timestamps:", t.duplicated().sum())



for df in dataframes:
    df["time(s)"] = df["time(s)"] - df["time(s)"][0]
    df["dt(s)"] = -df["time(s)"].diff(periods=-1)
    df["omega(rad/s)"] = -(2*math.pi / 360) * df["omega(deg/sec)"] 
    df["omega(rad/s)"] = df["omega(rad/s)"].ewm(span=5).mean()

    #df = df.ffill()
    df.fillna(0, inplace=True)

    # TODO add back in!
    spl = make_smoothing_spline(df["time(s)"], df["omega(rad/s)"], lam=0.1)
    df["omega(rad/s)"] = spl(df["time(s)"])


    #df = coordinate_transform_and_filter(df)
    df["alpha(rad/s/s)"] = -df["omega(rad/s)"].diff(periods=-1) / df["dt(s)"]
    df["alpha(rad/s/s)"] = df["alpha(rad/s/s)"].ewm(span=5).mean()
    df["alpha(rad/s/s)"] = df["alpha(rad/s/s)"].rolling(5).mean()
    #sos = signal.butter(10, 20, 'lp', fs=1000, output='sos')
    #df["alpha(rad/s/s)"] = signal.sosfilt(sos, df["alpha(rad/s/s)"])
    #df["alpha(rot/s/s)"] = df["alpha(rot/s/s)"].rolling(window=2, closed="left").mean()
    #print(df["omega(rad/s)"])
    #df = df.fillna(0)
    #df["omega(rad/s)"] = butterworth_filter(df["omega(rad/s)"], cutoff_freq, sampling_rate, order)
    #print(df["alpha(rad/s/s)"])
    #df = df.fillna(0)
    print("torque")
    df["T(Nm)"] = I_rotor * df["alpha(rad/s/s)"]
    print("torque done")

    #print(df["T(Nm)"])
print(df_27_kV)




In [ ]:


#print(min(df_28_kV["x_c"]))
print(min(df_28_kV["x"]))
print(max(df_28_kV["x"]))
#print(max(df_28_kV["x_c"]))
print(df_28_kV["theta"])
#plt.plot(df_28_kV["time(s)"], df_28_kV["x"])
#plt.plot(df_28_kV["time(s)"], df_28_kV["y"])
#plt.plot(df_28_kV["time(s)"], np.sqrt((df_28_kV["x"] - df_28_kV["x_c"])**2 + (df_28_kV["y"] - df_28_kV["y_c"])**2) , label="28.5 kV")
#plt.plot(df_28_kV["time(s)"], df_28_kV["r"], label="28.5 kV")
plt.plot(df_28_kV["time(s)"][0:50000], df_28_kV["theta"][0:50000], label="28.5 kV")

In [ ]:
index = 0

plt.plot(df_30_kV["time(s)"][0:]+50, rpm(df_30_kV["omega(rad/s)"])[0:], label="-30.0\,kV")
#plt.plot(df_29_kV["time(s)"][500:], -rpm(df_29_kV["omega(rad/s)"])[500:], label="-29.0\,kV")
plt.plot(df_28_kV["time(s)"][index:], rpm(df_28_kV["omega(rad/s)"])[index:], label="-28.0\,kV")
plt.plot(df_27_kV["time(s)"][index:], rpm(df_27_kV["omega(rad/s)"])[index:], label="-27.0\,kV")

plt.xlabel("Time (s)")

plt.ylabel("Speed (RPM)")
plt.ylim(0,1800)

#plt.axhline(1570)

plt.legend(loc = (0.5, 0.02))

plt.savefig("figures_20260817/speed_time_series.pdf", format="pdf")  
plt.savefig("figures_20260817/speed_time_series.png")  



In [ ]:
index = 0

plt.plot(df_29_kV["time(s)"][0:]-4, rpm(df_29_kV["omega(rad/s)"])[0:], label="-29.0\,kV")

plt.xlabel("Time (s)")

plt.ylabel("Speed (RPM)")
plt.ylim(-1800,0)

#plt.axhline(1570)

plt.legend(loc = (0.5, 0.02))

plt.savefig("figures_20260817/speed_time_series_reverse.pdf", format="pdf")  
plt.savefig("figures_20260817/speed_time_series_reverse.png")  

In [ ]:
fig, axs = plt.subplots(2)

index = 1250
index_end = 6000

axs[0].plot(np.nan, np.nan, label="-30.0\,kV")
axs[0].plot(df_28_kV["time(s)"][index:index_end]-5, rpm(df_28_kV["omega(rad/s)"])[index:index_end], label="-28.0\,kV")
axs[0].plot(df_27_kV["time(s)"][index:index_end]-5, rpm(df_27_kV["omega(rad/s)"])[index:index_end], label="-27.0\,kV")


axs[1].plot(df_30_kV["time(s)"][:]+45, rpm(df_30_kV["omega(rad/s)"])[:], label="-30.0\,kV")
axs[1].plot(df_28_kV["time(s)"][index_end:]-5, rpm(df_28_kV["omega(rad/s)"])[index_end:], label="-28.0\,kV")
axs[1].plot(df_27_kV["time(s)"][index_end:]-5, rpm(df_27_kV["omega(rad/s)"])[index_end:], label="-27.0\,kV")

plt.xlabel("Time (s)")

fig.supylabel("Speed (RPM)", fontsize=7, x=0.01)

axs[0].set_ylim(0, 1800)
axs[1].set_ylim(0, 1800)

#plt.axhline(1570)

plt.legend(loc='lower left',)

plt.savefig("figures_20260817/speed_time_series_chopped.pdf", format="pdf")  
plt.savefig("figures_20260817/speed_time_series_chopped.png")  


In [ ]:


index = 13640

plt.plot(df_27_kV["time(s)"][index:]-54.5, rpm(df_27_kV["omega(rad/s)"])[index:])
plt.xlabel("time (s)")

plt.ylabel("speed (rpm)")


index = 13640

plt.plot(df_28_kV["time(s)"][index:] - 54.4, rpm(df_28_kV["omega(rad/s)"])[index:])
plt.xlabel("time (s)")

plt.ylabel("speed (rpm)")



index = 2420

plt.plot(df_30_kV["time(s)"][index:]-4.9, rpm(df_30_kV["omega(rad/s)"])[index:])
plt.xlabel("time (s)")

plt.axvline(0)

plt.ylabel("speed (rpm)")

In [ ]:


def quad_fit(x, y):
    """
    Fit a quadratic function to data and return coefficients and R² value.
    
    Parameters:
    -----------
    x : array-like
        Independent variable data
    y : array-like
        Dependent variable data
    
    Returns:
    --------
    coeffs : tuple
        Quadratic coefficients (a, b, c) for y = ax² + bx + c
    r_squared : float
        R² value indicating goodness of fit
    """
    # Convert to numpy arrays
    x = np.array(x)
    y = np.array(y)
    
    # Fit quadratic polynomial (degree 2)
    coeffs = np.polyfit(x, y, 2)
    
    # Calculate predicted values
    y_pred = np.polyval(coeffs, x)
    
    # Calculate R² value
    ss_res = np.sum((y - y_pred)**2)  # Residual sum of squares
    ss_tot = np.sum((y - np.mean(y))**2)  # Total sum of squares
    r_squared = 1 - (ss_res / ss_tot)
    
    return tuple(coeffs), r_squared



plt.scatter(rpm(df_27_kV["omega(rad/s)"][13780:-200]), -df_27_kV["T(Nm)"][13780:-200] * 1000, alpha=0.5, s = 0.025, label="Spin-down trial 1")
plt.scatter(rpm(df_28_kV["omega(rad/s)"][13750:-200]), -df_28_kV["T(Nm)"][13750:-200] * 1000, alpha=0.5, s = 0.025, label="Spin-down trial 2")
plt.scatter(rpm(df_30_kV["omega(rad/s)"][2650:-200]), -df_30_kV["T(Nm)"][2650:-200] * 1000, alpha=0.5, s = 0.025, label="Spin-down trial 3")


#def fit(omega):
#    return -0.00003 - 0.0000000014 * ( omega) -  0.00000000009 * ( omega)**2



omega_list = np.linspace(0, 2000)

#coeffs, r_squared = quad_fit(rpm(df_28_kV["omega(rad/s)"][2500:-2000]), df_30_kV["T(Nm)"][2500:-2000])
coeffs, r_squared = quad_fit(rpm(df_28_kV["omega(rad/s)"][13750:-1000]), df_28_kV["T(Nm)"][13750:-1000])

def fit(omega):
    return coeffs[2] + coeffs[1] * ( omega) +  coeffs[0] * ( omega)**2


print(coeffs, r_squared)

print(coeffs[0])

plt.xlim(0,1550)
plt.ylim(0, 0.00025*1000)
plt.plot (omega_list, -1000*fit(omega_list), color="grey", linestyle='dashed', label= "Fit: " + format(-coeffs[0], ".2e") + "$\Omega^2$ + " + format(-coeffs[1], ".2e") + "$\Omega$ + \n" + "      "+ format(-coeffs[2], ".2e") +  ", $R^2$: "+ str(round(r_squared, 3)))
plt.ylabel("Load Torque (mN m)")
plt.xlabel("Speed (RPM)")


plt.rcParams["legend.markerscale"] = 20

leg = plt.legend()

for lh in leg.legend_handles: 
    lh.set_alpha(1)
plt.savefig("figures_20260817/Spindown.pdf", format="pdf")  
plt.savefig("figures_20260817/Spindown.png") 





In [ ]:
size=0.1
frequency=1

plt.plot(np.nan, np.nan, '-', color='none', label=' ') # fake plot to take space in the legend.

plt.plot(rpm(df_30_kV["omega(rad/s)"][100:2000:frequency]), (df_30_kV["T(Nm)"][100:2000:frequency] - fit(rpm(df_30_kV["omega(rad/s)"])[100:2000:frequency])) *  1000,  label="-30.0\,kV")
plt.plot(rpm(df_28_kV["omega(rad/s)"][0:10000:frequency]), (df_28_kV["T(Nm)"][:10000:frequency] - fit(rpm(df_28_kV["omega(rad/s)"])[:10000:frequency])) *  1000,label="-28.0\,kV")
plt.plot(rpm(df_27_kV["omega(rad/s)"][:10000:frequency]), (df_27_kV["T(Nm)"][:10000:frequency] - fit(rpm(df_27_kV["omega(rad/s)"])[:10000:frequency])) *  1000,  label="-27.0\,kV")

plt.plot (omega_list, -fit(omega_list) *  1000 , label="Load", color="gray")


eps0=8.854e-12
V_initial = 9000
eps_g=eps0
eps_r=eps0
R=0.030
L=0.054
G=0.0035
alpha=0.1
sigma = 4e-9

V_array = [30000, 28000, 27000]

linestyles = ['dashdot', 'dashed', 'dotted']

for i in range(len(V_array)):
    V = V_array[i]
    E_eff=(V-V_initial)/(2*R+2*G)
    tau = (eps_g + eps_r) / sigma
    omega = np.linspace(0, rad_per_sec(2000), 1000) # fix radians per second. 
    T_model = 2 * np.pi * L * R**2 * eps_g * E_eff**2 * ((np.sin(alpha)+  (omega * tau)* np.cos(alpha)) / (1 + (omega*tau)**2)) + \
        4 * np.pi * L * R**2 * ((eps_g*eps_r) / (eps_r+eps_g)) * E_eff**2 * ((np.sin(alpha)+  (omega * tau)* np.cos(alpha)) / (1 + (omega*tau)**2))
    plt.plot (rpm(omega), T_model * 1000 , label="Model: -" + str((V / 1000)) + "\,kV", color="gray", alpha = 0.3, linestyle = linestyles[i])




plt.xlim(0,2000)
plt.ylim(0,0.00032 * 1000)
plt.ylabel("Torque (mN m)")
plt.xlabel("Speed (RPM)")
plt.rcParams["legend.markerscale"] = 10
leg = plt.legend(ncol=2, loc='lower right', handlelength=1.2, columnspacing=0.7)
#leg = plt.legend(ncol=2, bbox_to_anchor=(0, 1), loc="lower left", fancybox=True, shadow=True)
#leg = plt.legend(ncol=2, bbox_to_anchor=(0.5, -0.15), loc="upper center", fancybox=True, shadow=True)
   
plt.savefig("figures_20260817/speed_torque_curve.pdf", format="pdf")  
plt.savefig("figures_20260817/speed_torque_curve.png")  

print("saved")

# Now plot the voltage dataset.

In [ ]:
#df_voltage = pd.read_csv('20260817_processed_video_csv/voltage_speed_data.csv', delimiter=",", names=["V", "deg/sec", "RPM"])

df_voltage = pd.read_csv('20260817_processed_video_csv/voltage_speed_data.csv', delimiter=",")

df_voltage = df_voltage.iloc[1:]
#df_voltage = df_voltage.reset_index(drop=True)
df_voltage = df_voltage.apply(pd.to_numeric, errors='coerce')

print(df_voltage)

res = stats.linregress(df_voltage["V"], df_voltage["speed(rpm)"])

print(f"R-squared: {res.rvalue**2:.6f}")

In [ ]:
plt.scatter(df_voltage["V"], df_voltage["speed(rpm)"], label = "Experimental Data", s=4, c="black")

#plt.scatter(df_voltage["V"], fit(df_voltage["speed(rpm)"]))

plt.plot(df_voltage["V"], res.intercept + res.slope*df_voltage["V"], 'grey', linestyle = 'dashed', label='Linear fit: a = '+ str(round(res.slope, 2)) +' RPM/V, \n b = ' + str(round(res.intercept, 2)) + ' V, $R^2$ = ' + str(round(res.rvalue**2, 3)))

#plt.plot(df_voltage["V"], poly1d_fn(df_voltage["V"]), color = "grey", linestyle='dashed')

plt.ylim(0,1800)
plt.xlim(17000,30000)
plt.xlabel("Drive Voltage (V)")
plt.ylabel("Steady State Speed (RPM)")
plt.rcParams["legend.markerscale"] = 1
plt.legend(ncol=1, loc='lower right')
#plt.savefig("Speed_vs_voltage.pdf", format="pdf")  





In [ ]:
from brokenaxes import brokenaxes



def force_ticks(event):
    """Force tick2line to be visible on every draw, but only on outer edges"""
    for i, ax in enumerate(bax.axs):
        # For x-axis: all axes get top ticks
        for tick in ax.xaxis.get_major_ticks():
            tick.tick2line.set_visible(True)
            tick.tick2line.set_markersize(4)
            tick.tick2line.set_markeredgewidth(0.8)
        for tick in ax.xaxis.get_minor_ticks():
            tick.tick2line.set_visible(True)
            tick.tick2line.set_markersize(2)
            tick.tick2line.set_markeredgewidth(0.6)
        
        # For y-axis: only the rightmost axis gets right ticks
        if i == len(bax.axs) - 1:  # Only the last (rightmost) axis
            for tick in ax.yaxis.get_major_ticks():
                tick.tick2line.set_visible(True)
                tick.tick2line.set_markersize(4)
                tick.tick2line.set_markeredgewidth(0.8)
            for tick in ax.yaxis.get_minor_ticks():
                tick.tick2line.set_visible(True)
                tick.tick2line.set_markersize(2)
                tick.tick2line.set_markeredgewidth(0.6)
        else:  # All other axes: explicitly hide right ticks
            for tick in ax.yaxis.get_major_ticks():
                tick.tick2line.set_visible(False)
            for tick in ax.yaxis.get_minor_ticks():
                tick.tick2line.set_visible(False)


fig = plt.figure()
bax = brokenaxes(
    xlims=((0, .1), (17000/1000, 31000/1000)),
    hspace=0.005
)


def speed_fit(voltage):
    return res.intercept + res.slope * voltage

bax.scatter(df_voltage["V"]/1000, df_voltage["speed(rpm)"], label = "Experimental Data", s=4, c="black")
bax.plot(df_voltage["V"]/1000, speed_fit(df_voltage["V"]), 'grey', linestyle = 'dashed', label='Linear fit: a = '+ str(round(res.slope, 2)) +'\,RPM/V, \n b = ' + str(round(res.intercept, 2)) + '\,V, $R^2$ = ' + str(round(res.rvalue**2, 3)))
bax.set_ylim(0,1800)
bax.set_xlabel("Drive Voltage (-kV)")
bax.set_ylabel("Steady State Speed (RPM)")
bax.legend(ncol=1, loc='lower right')

bax.axs[1].spines['right'].set_visible(True)
bax.axs[1].spines['top'].set_visible(True)

fig.canvas.mpl_connect('draw_event', force_ticks)
fig.canvas.draw()
fig.savefig("figures_20260817/Speed_vs_voltage.pdf", format="pdf")  
fig.savefig("figures_20260817/Speed_vs_voltage.png")  




In [ ]:
V = np.linspace(V_initial, 30000, 1000)


E_eff=(V-V_initial)/(2*R+2*G)

T_max = 2 * np.pi * L * R**2 * eps_g * E_eff**2 * (1+np.sin(alpha)) 

T_max = np.pi * L * R**2 * eps_g * E_eff**2 * (1+np.sin(alpha)) + 2* np.pi * L * R**2 * ((eps_g*eps_r) / (eps_r+eps_g)) * E_eff**2 


fig = plt.figure()
bax = brokenaxes(
    xlims=((0, .1), (8000/1000, 31000/1000)),
    hspace=0.005
)

bax.scatter(df_voltage["V"]/1000, -fit(df_voltage["speed(rpm)"])*1000, label = "Experimental Data", s=4, c="black")
bax.plot(V/1000, T_max*1000, 'grey', linestyle = 'dashed', label="Model") 
bax.set_ylim(0,0.00032*1000)
bax.set_xlabel("Drive Voltage (-kV)")
bax.set_ylabel("Steady Torque (mN m)", labelpad=22)
bax.legend(ncol=2, loc='upper left')

bax.axs[1].spines['right'].set_visible(True)
bax.axs[1].spines['top'].set_visible(True)

fig.canvas.mpl_connect('draw_event', force_ticks)
fig.canvas.draw()
plt.savefig("figures_20260817/torque_vs_voltage.pdf", format="pdf")  
plt.savefig("figures_20260817/torque_vs_voltage.png")  



In [ ]:


fig = plt.figure()
bax = brokenaxes(
    xlims=((0, .1), (8000/1000, 31000/1000)),
    hspace=0.005
)




bax.scatter(df_voltage["V"]/1000, -fit(df_voltage["speed(rpm)"]) * rad_per_sec(df_voltage["speed(rpm)"]), label = "Experimental Data", s=4, c="black")

bax.set_ylim(0,)
bax.set_xlabel("Drive Voltage (-kV)")
bax.set_ylabel("Motor Power (W)", labelpad=22)
bax.legend(ncol=2, loc='upper left')

bax.axs[1].spines['right'].set_visible(True)
bax.axs[1].spines['top'].set_visible(True)

fig.canvas.mpl_connect('draw_event', force_ticks)
fig.canvas.draw()
plt.savefig("figures_20260817/motor_power_vs_voltage.pdf", format="pdf")  
plt.savefig("figures_20260817/motor_power_vs_voltage.png")  


In [ ]:
df_current = pd.read_csv('20260817_processed_video_csv/current_data.csv', delimiter=",")

df_current = df_current.iloc[1:]
df_voltage = df_voltage.reset_index(drop=True)
df_current = df_current.apply(pd.to_numeric, errors='coerce')

print(df_current)

In [ ]:

plt.clf()
fig = plt.figure()

bax = brokenaxes(
    xlims=((0, .1), (8000/1000, 31000/1000)),
    hspace=0.005
)




bax.scatter(df_current['V']/1000, 1000 * df_current['V_shunt_trial1'] / df_current['R(ohms)'], label="trial 1")
bax.scatter(df_current['V']/1000, 1000 * df_current['V_shunt_trial2'] / df_current['R(ohms)'], label="trial 2")


df_current['I_1'] = df_current['V_shunt_trial1'] / df_current['R(ohms)']
df_current['I_2'] = df_current['V_shunt_trial2'] / df_current['R(ohms)']

print(df_current)
df_current["I_avg"] = df_current[['I_1', 'I_2']].mean(axis=1, skipna=False)

print(df_current)


bax.set_ylim(0,)
bax.set_xlabel("Drive Voltage (-kV)")
bax.set_ylabel("Motor Current (mA)", labelpad=20)
bax.legend(ncol=1, loc='upper left')

bax.axs[1].spines['right'].set_visible(True)
bax.axs[1].spines['top'].set_visible(True)



fig.canvas.mpl_connect('draw_event', force_ticks)
fig.canvas.draw()




plt.savefig("figures_20260817/motor_current_vs_voltage.pdf", format="pdf")  
plt.savefig("figures_20260817/motor_current_vs_voltage.png")  







In [ ]:
#plt.clf()
fig = plt.figure()
bax = brokenaxes(
    xlims=((0, .1), (8000/1000, 31000/1000)),
    hspace=0.005
)


#current_avg = np.mean(df_current['V_shunt_trial1'],df_current['V_shunt_trial2'] )
#print(current_avg)

bax.scatter(df_current['V']/1000, df_current['V'] *(df_current['V_shunt_trial1'] / df_current['R(ohms)']), label="trial 1")

bax.scatter(df_current['V']/1000, df_current['V'] *(df_current['V_shunt_trial2'] / df_current['R(ohms)']), label="trial 2")





bax.set_ylim(0,)
bax.set_xlabel("Drive Voltage (-kV)")
bax.set_ylabel("Motor Input Power (W)", labelpad=10)
bax.legend(ncol=1, loc='upper left')

bax.axs[1].spines['right'].set_visible(True)
bax.axs[1].spines['top'].set_visible(True)



fig.canvas.mpl_connect('draw_event', force_ticks)
fig.canvas.draw()




plt.savefig("figures_20260817/motor_electrical_power_vs_voltage.pdf", format="pdf")  
plt.savefig("figures_20260817/motor_electrical_power_vs_voltage.png")  

In [ ]:

#plt.clf()
fig = plt.figure()
bax = brokenaxes(
    xlims=((0, .1), (8000/1000, 31000/1000)),
    hspace=0.005
)

bax.scatter(df_current['V']/1000, 100 * (-fit(speed_fit(df_current['V'])) * rad_per_sec(speed_fit(df_current['V']))) / (df_current['V']* df_current['I_avg']))


bax.set_ylim(0,1.2)
bax.set_xlabel("Drive Voltage (-kV)")
bax.set_ylabel("\% Motor efficiency", labelpad=20)
bax.legend(ncol=2, loc='upper left')

bax.axs[1].spines['right'].set_visible(True)
bax.axs[1].spines['top'].set_visible(True)



fig.canvas.mpl_connect('draw_event', force_ticks)
fig.canvas.draw()






plt.savefig("figures_20260817/motor_current_vs_voltage.pdf", format="pdf")  
plt.savefig("figures_20260817/motor_current_vs_voltage.png")  

## Calculating torque density.


In [ ]:


peak_torque = 0.25e-3 # N m
total_mass = 45e-3 # kg
active_mass = 0.75e-3 + 1.40e-3# kg
peak_power = 0.05 # W

endcap_side_length = 0.033 # m
A_endcap = 2 * (1 + np.sqrt(2)) * endcap_side_length**2 # m^2
height_stowed = 0.016 # m
height_deployed = 0.062 # m
V_stowed = height_stowed * A_endcap
V_deployed = height_deployed * A_endcap

print("deployed volume: " + str(V_deployed))
print("stowed volume: " + str(V_stowed))


torque_per_volume_stowed = peak_torque / V_stowed
torque_per_volume_deployed = peak_torque / V_deployed
print("torque per volume deployed: " + str(torque_per_volume_deployed))
print("torque per volume stowed: " + str(torque_per_volume_stowed))

torque_per_total_mass = peak_torque / total_mass
print("torque per total mass: " + str(torque_per_total_mass))
torque_per_active_mass = peak_torque / active_mass
print("torque per active mass: " + str(torque_per_active_mass))


peak_power_density_total_mass = peak_power / total_mass
print("power per total mass: " + str(peak_power_density_total_mass))
peak_power_density_active = peak_power / active_mass
print("power per active mass: " + str(peak_power_density_active))

